In [1]:
!pip install -r requirements.txt

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
    --------------------------------------- 0.3/18.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.2 MB 1.2 MB/s eta 0:00:15
   - -------------------------------------- 0.8/18.2 MB 1.3 MB/s eta 0:00:13
   -- ------------------------------------- 1.3/18.2 MB 1.4 MB/s eta 0:00:13
   --- ------------------------------------ 1.6/18.2 MB 1.4 MB/s eta 0:00:12
   --------- ------------------------------ 4.2/18.2 MB 3.2 MB/s eta 0:00:05
   ------------------- -------------------- 8.7/18.2 MB 5.8 MB/s eta 0:00:02
   --------------------------- ------------ 12.3/18.2 MB 7.3 MB/s eta 0:00:01
   ----------------------------------- ---- 16.0/18.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 18.2/18.2 MB 8.7 MB/s  0:00:02


### FAISS Vector Store (Simple Definition)

FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta that is optimized for fast similarity search on large collections of vectors (embeddings).
- Used for efficient similarity search and clustering of dense vectors.
- Contains Algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM.
- It also contains supporting code for evaluation and parameter tuning.

All vecotr DBs: 
https://python.langchain.com/docs/integrations/vectorstores/

In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings


loader = TextLoader("text.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(separator = "\n", chunk_size = 100, chunk_overlap = 20)
docs = text_splitter.split_documents(documents)
docs

Created a chunk of size 105, which is longer than the specified 100
Created a chunk of size 105, which is longer than the specified 100
Created a chunk of size 103, which is longer than the specified 100


[Document(metadata={'source': 'text.txt'}, page_content='The Taj Mahal is a white marble mausoleum located in Agra, India.'),
 Document(metadata={'source': 'text.txt'}, page_content='It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his wife Mumtaz Mahal.'),
 Document(metadata={'source': 'text.txt'}, page_content='The Taj Mahal combines elements from Islamic, Persian, Ottoman Turkish, and Indian architectural styles.'),
 Document(metadata={'source': 'text.txt'}, page_content='It is considered one of the most beautiful buildings in the world and is a UNESCO World Heritage Site.'),
 Document(metadata={'source': 'text.txt'}, page_content='Millions of tourists visit the Taj Mahal every year, making it one of the most famous landmarks in India.')]

In [16]:
embeddings = OllamaEmbeddings(model = "embeddinggemma")

db = FAISS.from_documents(
  documents= docs,
  embedding= embeddings
)
db

C:\Users\ayush\AppData\Local\Temp\ipykernel_32084\3455424275.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model = "embeddinggemma")


In [24]:
query = "Where is Taj Mahal?"

docs = db.similarity_search(query)
docs

[Document(id='60107da8-705e-44e3-a2e2-3b25bd1579dc', metadata={'source': 'text.txt'}, page_content='The Taj Mahal is a white marble mausoleum located in Agra, India.'),
 Document(id='41e7baa2-1ccd-41de-9dd1-3c3797056cff', metadata={'source': 'text.txt'}, page_content='Millions of tourists visit the Taj Mahal every year, making it one of the most famous landmarks in India.'),
 Document(id='3a4209c3-adcc-48c0-891b-186ec7e612d0', metadata={'source': 'text.txt'}, page_content='The Taj Mahal combines elements from Islamic, Persian, Ottoman Turkish, and Indian architectural styles.'),
 Document(id='be6eb7f9-0fad-4dd4-91d1-82917b33a74b', metadata={'source': 'text.txt'}, page_content='It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his wife Mumtaz Mahal.')]

In [19]:
docs[0].page_content

'The Taj Mahal is a white marble mausoleum located in Agra, India.'

In [25]:
query = "Who built Taj mahal?"

docs = db.similarity_search(query)
docs

[Document(id='60107da8-705e-44e3-a2e2-3b25bd1579dc', metadata={'source': 'text.txt'}, page_content='The Taj Mahal is a white marble mausoleum located in Agra, India.'),
 Document(id='41e7baa2-1ccd-41de-9dd1-3c3797056cff', metadata={'source': 'text.txt'}, page_content='Millions of tourists visit the Taj Mahal every year, making it one of the most famous landmarks in India.'),
 Document(id='3a4209c3-adcc-48c0-891b-186ec7e612d0', metadata={'source': 'text.txt'}, page_content='The Taj Mahal combines elements from Islamic, Persian, Ottoman Turkish, and Indian architectural styles.'),
 Document(id='be6eb7f9-0fad-4dd4-91d1-82917b33a74b', metadata={'source': 'text.txt'}, page_content='It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his wife Mumtaz Mahal.')]

#### Actually above it is just returning us the similar vectors to our query, here as we have chunks so it will return similar vector chunks to our query.

## As a Retriever

#### We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other Langchain methods, which largely work with retrivers.

In [27]:
retriever = db.as_retriever()

query = "Where is Taj Mahal?"
retriever.invoke(query)  ## same similar vector chunks we get as above by similarity search

[Document(id='60107da8-705e-44e3-a2e2-3b25bd1579dc', metadata={'source': 'text.txt'}, page_content='The Taj Mahal is a white marble mausoleum located in Agra, India.'),
 Document(id='41e7baa2-1ccd-41de-9dd1-3c3797056cff', metadata={'source': 'text.txt'}, page_content='Millions of tourists visit the Taj Mahal every year, making it one of the most famous landmarks in India.'),
 Document(id='3a4209c3-adcc-48c0-891b-186ec7e612d0', metadata={'source': 'text.txt'}, page_content='The Taj Mahal combines elements from Islamic, Persian, Ottoman Turkish, and Indian architectural styles.'),
 Document(id='be6eb7f9-0fad-4dd4-91d1-82917b33a74b', metadata={'source': 'text.txt'}, page_content='It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his wife Mumtaz Mahal.')]

### Similarity Search with score

#### There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents, but also the distance score of the query to them. The returned distance score is L2 (Manhattan) distance. Therefore, a lower score is better.

In [28]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='60107da8-705e-44e3-a2e2-3b25bd1579dc', metadata={'source': 'text.txt'}, page_content='The Taj Mahal is a white marble mausoleum located in Agra, India.'),
  np.float32(145064.9)),
 (Document(id='41e7baa2-1ccd-41de-9dd1-3c3797056cff', metadata={'source': 'text.txt'}, page_content='Millions of tourists visit the Taj Mahal every year, making it one of the most famous landmarks in India.'),
  np.float32(159530.66)),
 (Document(id='3a4209c3-adcc-48c0-891b-186ec7e612d0', metadata={'source': 'text.txt'}, page_content='The Taj Mahal combines elements from Islamic, Persian, Ottoman Turkish, and Indian architectural styles.'),
  np.float32(180437.52)),
 (Document(id='be6eb7f9-0fad-4dd4-91d1-82917b33a74b', metadata={'source': 'text.txt'}, page_content='It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his wife Mumtaz Mahal.'),
  np.float32(198816.28))]

In [29]:
## Also,
query = "Who built Taj mahal?"

embedding_vector = embeddings.embed_query(query)
embedding_vector

[-90.73477172851562,
 -18.733449935913086,
 29.207040786743164,
 16.572053909301758,
 8.150077819824219,
 9.110034942626953,
 -32.46784973144531,
 24.90355682373047,
 6.121708869934082,
 -32.62382507324219,
 -9.33955192565918,
 -32.023555755615234,
 37.822608947753906,
 14.793668746948242,
 48.004966735839844,
 11.52881908416748,
 2.549072742462158,
 -20.822614669799805,
 -43.27302932739258,
 -1.455582618713379,
 9.870037078857422,
 8.292530059814453,
 -23.91037368774414,
 -8.844514846801758,
 -17.38805389404297,
 -3.8059425354003906,
 22.72352409362793,
 -13.98601245880127,
 3.3501815795898438,
 17.086254119873047,
 15.998771667480469,
 -30.177602767944336,
 1.1573638916015625,
 10.831901550292969,
 3.308145523071289,
 40.56787109375,
 5.995111465454102,
 -32.43758773803711,
 -5.2131500244140625,
 10.402848243713379,
 -33.752662658691406,
 30.289146423339844,
 5.495330810546875,
 -18.073747634887695,
 -6.6424713134765625,
 8.236305236816406,
 -21.145076751708984,
 3.7737255096435547,


In [30]:
db.similarity_search_by_vector(embedding_vector)

[Document(id='60107da8-705e-44e3-a2e2-3b25bd1579dc', metadata={'source': 'text.txt'}, page_content='The Taj Mahal is a white marble mausoleum located in Agra, India.'),
 Document(id='41e7baa2-1ccd-41de-9dd1-3c3797056cff', metadata={'source': 'text.txt'}, page_content='Millions of tourists visit the Taj Mahal every year, making it one of the most famous landmarks in India.'),
 Document(id='3a4209c3-adcc-48c0-891b-186ec7e612d0', metadata={'source': 'text.txt'}, page_content='The Taj Mahal combines elements from Islamic, Persian, Ottoman Turkish, and Indian architectural styles.'),
 Document(id='be6eb7f9-0fad-4dd4-91d1-82917b33a74b', metadata={'source': 'text.txt'}, page_content='It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his wife Mumtaz Mahal.')]

In [32]:
## Saving and Loading Embeddings
db.save_local("faiss-index")

In [35]:
new_db = FAISS.load_local("faiss-index", embeddings= embeddings, allow_dangerous_deserialization=True)
new_db

In [36]:
docs = new_db.similarity_search(query)
docs

[Document(id='60107da8-705e-44e3-a2e2-3b25bd1579dc', metadata={'source': 'text.txt'}, page_content='The Taj Mahal is a white marble mausoleum located in Agra, India.'),
 Document(id='41e7baa2-1ccd-41de-9dd1-3c3797056cff', metadata={'source': 'text.txt'}, page_content='Millions of tourists visit the Taj Mahal every year, making it one of the most famous landmarks in India.'),
 Document(id='3a4209c3-adcc-48c0-891b-186ec7e612d0', metadata={'source': 'text.txt'}, page_content='The Taj Mahal combines elements from Islamic, Persian, Ottoman Turkish, and Indian architectural styles.'),
 Document(id='be6eb7f9-0fad-4dd4-91d1-82917b33a74b', metadata={'source': 'text.txt'}, page_content='It was commissioned in 1632 by the Mughal emperor Shah Jahan to house the tomb of his wife Mumtaz Mahal.')]